# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de seguimiento de inventario

Este notebook trabaja **solo** con `andinalog_inventory_tracking.csv`. Lee la capa Bronze desde `datasets/AndinaLog_03B_Bronce/`, detecta problemas y conserva sus campos originales. No convierte unidades, no imputa ni corrige datos. Marca `-999` en cantidad como valor centinela inválido y lo deja en cuarentena. El tratamiento de los casos recuperables corresponde al notebook 2.

Cada ejecución reemplaza cuatro archivos en `S4/andinalog_inventory_tracking/notebook1/salidas/`:

1. `andinalog_inventory_tracking_diagnosticado.csv`: todas las filas, los campos originales y solo `fila_bronze`, `en_cuarentena` y `columnas_con_problemas`.
2. `andinalog_inventory_tracking_problemas.csv`: una fila por problema, con columna, código estable y evidencia.
3. `andinalog_inventory_tracking_cuarentena.csv`: extracto informativo de las filas marcadas.
4. `andinalog_inventory_tracking_reporte_calidad.csv`: conteos y huella SHA-256 del CSV de origen.

## 1 · Configuración y origen

En local, ejecuta el notebook desde cualquier carpeta dentro del proyecto. En Colab, monta Drive, selecciona `ENTORNO = "drive"` y ajusta `RUTA_PROYECTO_DRIVE` a la carpeta que contiene `datasets/` y `S4/`. Las salidas van a `S4/andinalog_inventory_tracking/notebook1/salidas/` en ese mismo entorno.

In [24]:
from pathlib import Path
import hashlib
import os
import re
import tempfile
import pandas as pd

ENTORNO = "auto"  # "auto", "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
NOMBRE_CSV = "andinalog_warehouse_costs.csv"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-INV-diagnostico-v1"

COLUMNAS_ORIGINALES = [
    "centro_distribucion",
    "rotacion_stock_dias",
    "perdida_mermas_bob",
    "periodo_mes",
    "costo_almacenamiento_mensual_bob"
]

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / CARPETA_DATASETS).is_dir() and (carpeta / "S4").is_dir():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz del proyecto; ejecuta dentro de practicasNotebookColab.")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser 'auto', 'local' o 'drive'")
    bronze = raiz / "datasets" / CARPETA_DATASETS / NOMBRE_CSV
    salidas = raiz / "S4" / "andinalog_warehouse_costs" / "notebook1" / "salidas"
    if not bronze.is_file():
        raise FileNotFoundError(f"No se encontró el CSV Bronze: {bronze}")
    return bronze, salidas

RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
print("Bronze:", RUTA_BRONZE)
print("Salidas:", DIRECTORIO_SALIDAS)


Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_warehouse_costs.csv
Salidas: c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_warehouse_costs\notebook1\salidas


## 2 · Carga y contrato

La lectura mantiene todas las columnas como texto y los vacíos como cadenas vacías. Las conversiones numéricas y de fecha usadas para comprobar errores son temporales: no se exportan como valores transformados.

In [25]:
def cargar_bronze(ruta):
    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()
    df = pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return df, huella

def validar_esquema(df):
    if list(df.columns) != COLUMNAS_ORIGINALES:
        faltantes = sorted(set(COLUMNAS_ORIGINALES) - set(df.columns))
        extras = sorted(set(df.columns) - set(COLUMNAS_ORIGINALES))
        raise ValueError(f"Esquema inesperado. Faltantes: {faltantes}; extras: {extras}; orden: {list(df.columns)}")
    if not df.columns.is_unique:
        raise ValueError("Hay nombres de columnas duplicados")
    return df

df_bronze, HASH_BRONZE = cargar_bronze(RUTA_BRONZE)
validar_esquema(df_bronze)
print(f"Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas")
print("SHA-256:", HASH_BRONZE)
display(df_bronze.head())

Bronze: 6 filas × 5 columnas
SHA-256: 60ee80d8be911f2a7c69527bf0dce2cae6d007956623f8617b11917170f125a1


,centro_distribucion,rotacion_stock_dias,perdida_mermas_bob,periodo_mes,costo_almacenamiento_mensual_bob
0,Cochabamba,17.25,395488.73,2026-08,140000
1,La Paz,17.51,405070.14,2026-08,180000
2,Oruro,17.11,429637.2,2026-08,95000
3,Santa Cruz,17.57,395202.88,2026-08,220000
4,Tarija,17.92,400504.77,2026-08,110000


## 3 · Catálogo y reglas de diagnóstico

Los códigos son estables para que el notebook 2 pueda reconocer el problema exacto. `columna_afectada` puede nombrar una sola columna o una clave compuesta, como `tracking_id+timestamp`. Las reglas detectan; no deciden todavía cómo corregir.

`VALOR_CENTINELA` identifica `-999` en `cantidad`. No se sustituye por un valor estimado: permanece en cuarentena hasta contar con un registro verificado.

In [26]:
CATALOGO_PROBLEMAS = pd.DataFrame([
    ("centro_distribucion", "FALTANTE", "Centro vacío"),
    ("centro_distribucion", "VALOR_NO_RECONOCIDO", "Centro fuera del dominio operativo declarado"),
    
    ("rotacion_stock_dias", "FALTANTE", "Campo vacío"),
    ("rotacion_stock_dias", "NO_NUMERICA", "Valor no convertible a número"),
    ("rotacion_stock_dias", "FUERA_RANGO", "Valor fuera de límites válidos (negativo o nulo)"),
    
    ("perdida_mermas_bob", "FALTANTE", "Campo vacío"),
    ("perdida_mermas_bob", "NO_NUMERICA", "Valor no convertible a número"),
    ("perdida_mermas_bob", "FUERA_RANGO", "Valor fuera de límites válidos (negativo)"),
    
    ("periodo_mes", "FECHA_INVALIDA", "No cumple AAAA-MM o no existe en el calendario"),
    ("periodo_mes+centro_distribucion", "DUPLICADO", "Clave repetida; se marca la aparición posterior"),
    
    ("costo_almacenamiento_mensual_bob", "FALTANTE", "Campo vacío"),
    ("costo_almacenamiento_mensual_bob", "NO_NUMERICA", "Valor no convertible a número"),
    ("costo_almacenamiento_mensual_bob", "FUERA_RANGO", "Valor fuera de límites válidos (negativo)")
], columns=["columna_afectada", "codigo_error", "criterio"])

display(CATALOGO_PROBLEMAS)

def texto(df, columna):
    return df[columna].astype("string").str.strip()

def registrar_problema(df, mascara, columna, codigo, evidencia=None):
    mascara = mascara.fillna(False).astype(bool)
    filas = df.loc[mascara, ["fila_bronze"]].copy()
    filas["columna_afectada"] = columna
    filas["codigo_error"] = codigo
    if evidencia is None:
        evidencia = df[columna] if columna in df.columns else pd.Series("", index=df.index, dtype="string")
    filas["valor_original"] = evidencia.loc[mascara].astype("string").to_numpy()
    return filas

def detectar_centro(df):
    original = df["centro_distribucion"].astype("string")
    limpio = original.str.strip()
    permitidos = {"La Paz", "Cochabamba", "Santa Cruz", "Oruro", "Tarija"}
    return [
        registrar_problema(df, limpio.eq(""), "centro_distribucion", "FALTANTE", original),
        registrar_problema(df, limpio.ne("") & ~original.isin(permitidos), "centro_distribucion", "VALOR_NO_RECONOCIDO", original),
    ]

def detectar_periodos_y_duplicados(df):
    fecha_original = df["periodo_mes"].astype("string")
    fecha_limpia = fecha_original.str.strip()
    formato = fecha_limpia.str.fullmatch(r"\d{4}-\d{2}").fillna(False)
    fecha = pd.to_datetime(fecha_limpia + "-01", format="%Y-%m-%d", errors="coerce")
    duplicada = (texto(df, "centro_distribucion") + "+" + fecha_limpia).duplicated(keep="first")
    return [
        registrar_problema(df, ~formato | fecha.isna(), "periodo_mes", "FECHA_INVALIDA", fecha_original),
        registrar_problema(df, duplicada, "periodo_mes+centro_distribucion", "DUPLICADO", texto(df, "centro_distribucion") + "+" + fecha_limpia),
    ]

def detectar_numero(df, columna, minimo=0, codigo_limite="FUERA_RANGO"):
    original = df[columna].astype("string")
    limpio = original.str.strip()
    numero = pd.to_numeric(limpio, errors="coerce")
    return [
        registrar_problema(df, limpio.eq(""), columna, "FALTANTE", original),
        registrar_problema(df, limpio.ne("") & numero.isna(), columna, "NO_NUMERICA", original),
        registrar_problema(df, numero.notna() & numero.lt(minimo), columna, codigo_limite, original),
    ]

def diagnosticar(df_bronze):
    principal = df_bronze.copy(deep=True)
    principal.insert(0, "fila_bronze", range(1, len(principal) + 1))
    hallazgos = []
    hallazgos += detectar_centro(principal)
    hallazgos += detectar_periodos_y_duplicados(principal)
    hallazgos += detectar_numero(principal, "rotacion_stock_dias", minimo=0, codigo_limite="FUERA_RANGO")
    hallazgos += detectar_numero(principal, "perdida_mermas_bob", minimo=0, codigo_limite="FUERA_RANGO")
    hallazgos += detectar_numero(principal, "costo_almacenamiento_mensual_bob", minimo=0, codigo_limite="FUERA_RANGO")
    problemas = pd.concat(hallazgos, ignore_index=True)
    problemas = problemas.sort_values(["fila_bronze", "columna_afectada", "codigo_error"], kind="stable").reset_index(drop=True)
    problemas["version_diagnostico"] = VERSION_DIAGNOSTICO
    columnas_por_fila = problemas.groupby("fila_bronze")["columna_afectada"].agg(lambda valores: "|".join(dict.fromkeys(valores)))
    principal["columnas_con_problemas"] = principal["fila_bronze"].map(columnas_por_fila).fillna("")
    principal["en_cuarentena"] = principal["columnas_con_problemas"].ne("")
    return principal, problemas

df_diagnosticado, df_problemas = diagnosticar(df_bronze)
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()
print(f"Principal: {len(df_diagnosticado):,}; problemas: {len(df_problemas):,}; filas en cuarentena: {len(df_cuarentena):,}")
display(df_problemas.groupby(["columna_afectada", "codigo_error"]).size().rename("filas").reset_index())


,columna_afectada,codigo_error,criterio
0,centro_distribucion,FALTANTE,Centro vacío
1,centro_distribucion,VALOR_NO_RECONOCIDO,Centro fuera del dominio operativo declarado
2,rotacion_stock_dias,FALTANTE,Campo vacío
3,rotacion_stock_dias,NO_NUMERICA,Valor no convertible a número
4,rotacion_stock_dias,FUERA_RANGO,Valor fuera de límites válidos (negativo o nulo)
5,perdida_mermas_bob,FALTANTE,Campo vacío
6,perdida_mermas_bob,NO_NUMERICA,Valor no convertible a número
7,perdida_mermas_bob,FUERA_RANGO,Valor fuera de límites válidos (negativo)
8,periodo_mes,FECHA_INVALIDA,No cumple AAAA-MM o no existe en el calendario
9,periodo_mes+centro_distribucion,DUPLICADO,Clave repetida; se marca la aparición posterior


Principal: 6; problemas: 2; filas en cuarentena: 1


,columna_afectada,codigo_error,filas
0,periodo_mes+centro_distribucion,DUPLICADO,1
1,rotacion_stock_dias,FALTANTE,1


## 4 · Reporte y comprobaciones antes de exportar

El reporte registra la huella SHA-256 para reconocer la versión exacta del CSV de origen. Los conteos de problemas pueden superar el número de filas en cuarentena porque una fila puede tener varios hallazgos.

In [27]:
def construir_reporte(df_bronze, principal, problemas, ruta, huella):
    conteos = problemas.groupby(["columna_afectada", "codigo_error"]).size()
    datos = [
        ("archivo_bronze", ruta.name),
        ("sha256_bronze", huella),
        ("version_diagnostico", VERSION_DIAGNOSTICO),
        ("filas_bronze", len(df_bronze)),
        ("filas_diagnosticadas", len(principal)),
        ("filas_en_cuarentena", int(principal["en_cuarentena"].sum())),
        ("filas_sin_cuarentena", int((~principal["en_cuarentena"]).sum())),
        ("problemas_detectados", len(problemas)),
    ]
    datos += [(f"{col}:{codigo}", int(total)) for (col, codigo), total in conteos.items()]
    return pd.DataFrame(datos, columns=["metrica", "valor"])

def validar_resultados(df_bronze, principal, problemas, cuarentena, reporte):
    assert list(principal.columns) == ["fila_bronze", *COLUMNAS_ORIGINALES, "columnas_con_problemas", "en_cuarentena"]
    pd.testing.assert_frame_equal(principal[COLUMNAS_ORIGINALES], df_bronze[COLUMNAS_ORIGINALES])
    assert len(principal) == len(df_bronze)
    assert principal["fila_bronze"].is_unique
    assert len(cuarentena) == int(principal["en_cuarentena"].sum())
    assert problemas["fila_bronze"].isin(principal["fila_bronze"]).all()
    assert problemas[["columna_afectada", "codigo_error"]].apply(tuple, axis=1).isin(
        CATALOGO_PROBLEMAS[["columna_afectada", "codigo_error"]].apply(tuple, axis=1)
    ).all()
    assert set(problemas["fila_bronze"]) == set(cuarentena["fila_bronze"])
    assert len(reporte) >= 8

# --- Ejecución final ---


reporte_calidad = construir_reporte(df_bronze, df_diagnosticado, df_problemas, RUTA_BRONZE, HASH_BRONZE)
validar_resultados(df_bronze, df_diagnosticado, df_problemas, df_cuarentena, reporte_calidad)

display(reporte_calidad)
print("Comprobaciones previas a la exportación: correctas")

,metrica,valor
0,archivo_bronze,andinalog_warehouse_costs.csv
1,sha256_bronze,60ee80d8be911f2a7c69527bf0dce2cae6d007956623f8...
2,version_diagnostico,GIAD-M3-S4-INV-diagnostico-v1
3,filas_bronze,6
4,filas_diagnosticadas,6
5,filas_en_cuarentena,1
6,filas_sin_cuarentena,5
7,problemas_detectados,2
8,periodo_mes+centro_distribucion:DUPLICADO,1
9,rotacion_stock_dias:FALTANTE,1


Comprobaciones previas a la exportación: correctas


## 5 · Exportación reproducible

Los cuatro CSV se escriben primero como archivos temporales en `S4/andinalog_inventory_tracking/notebook1/salidas/` y se reemplazan con el mismo nombre al final. El CSV Bronze nunca se sobrescribe. Si se vuelve a ejecutar con la misma fuente y reglas, las salidas se actualizan en lugar de acumular versiones antiguas.

In [28]:
def exportar_salidas(directorio, tablas, ruta_bronze, huella_inicial):
    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:
        raise RuntimeError("El CSV Bronze cambió durante la ejecución; no se exportarán resultados")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_inv_", dir=directorio,
                                             encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

tablas_salida = {
    "andinalog_warehouse_costs_diagnosticado.csv": df_diagnosticado,
    "andinalog_warehouse_costs_problemas.csv": df_problemas,
    "andinalog_warehouse_costs_cuarentena.csv": df_cuarentena,
    "andinalog_warehouse_costs_reporte_calidad.csv": reporte_calidad,
}
rutas_creadas = exportar_salidas(DIRECTORIO_SALIDAS, tablas_salida, RUTA_BRONZE, HASH_BRONZE)
for ruta in rutas_creadas:
    print(ruta)
print("Bronze intacta; salidas anteriores reemplazadas")

c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_warehouse_costs\notebook1\salidas\andinalog_warehouse_costs_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_warehouse_costs\notebook1\salidas\andinalog_warehouse_costs_problemas.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_warehouse_costs\notebook1\salidas\andinalog_warehouse_costs_cuarentena.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_warehouse_costs\notebook1\salidas\andinalog_warehouse_costs_reporte_calidad.csv
Bronze intacta; salidas anteriores reemplazadas


## Siguiente etapa

El notebook 2 leerá el archivo diagnosticado y el detalle de problemas. El informe de S4 justifica tratamientos posibles, pero ninguna regla de curación está aprobada automáticamente por este diagnóstico. Una fila saldrá de cuarentena solo cuando todos sus problemas hayan sido resueltos y validados.